# ARGUS on Google Colab — bootstrap test

Smallest possible test that ARGUS can run inside a Colab notebook. After running cells 2–4 (the bootstrap), the standard `%%ask` magic should work on the test cell at the bottom.

**Prerequisites:**
- A Google account with Colab access
- A ZAI API key (https://z.ai — much cheaper than OpenAI for development), saved as a Colab Secret named `ZAI_API_KEY`
  - Open the 🔑 sidebar (key icon, left edge of Colab) → Add new secret → Name: `ZAI_API_KEY` → Value: your key → toggle "Notebook access" on
  - To use a different provider (OpenAI, Anthropic, NRP, OpenRouter, Nvidia, etc.), edit the `config.toml` block in cell 3 — set `default`, `base_url`, and `api_key_env` to whatever you prefer. ARGUS reads `api_key_env` at runtime, so any provider works without code changes.

**Goal:** the final `%%ask` cell answers "what date is today?" using ARGUS's agent loop. If that works, the bootstrap pattern is sound and we can extend with MCP, skills, etc.

In [ ]:
# 1. Install ARGUS dependencies
# (deepagents-code requires Python 3.11+; Colab's default kernel ships Python 3.11)
%pip install -q deepagents-code==0.1.10 langchain-mcp-adapters nest_asyncio \
    folium geopandas ipyleaflet ipywidgets matplotlib rasterio leafmap plotly \
    pypdf openpyxl tomli-w

In [ ]:
# 2. Download ARGUS source files
# Pull from 'main' during experimentation so fixes land immediately;
# switch to a release tag (e.g. v1.4.2) once the bootstrap pattern is stable.
REF = 'main'
BASE = f'https://raw.githubusercontent.com/klinucsd/sage/{REF}'

!curl -sLo /content/sage_magic.py {BASE}/sage_magic.py
!curl -sLo /content/sage_kernel_backend.py {BASE}/sage_kernel_backend.py

import sys
sys.path.insert(0, '/content')

!ls -la /content/sage_magic.py /content/sage_kernel_backend.py

In [ ]:
# 3. Set up config.toml for deepagents-code, defaulting to ZAI GLM-5
# ZAI's hosted GLM-5 is much cheaper than OpenAI for development; swap the
# block below if you prefer a different provider. ARGUS's API-key check
# reads `api_key_env` from this file at runtime, so any provider works.
import os, pathlib, textwrap

deepagents_dir = pathlib.Path('~/.deepagents').expanduser()
(deepagents_dir / 'agent' / 'skills').mkdir(parents=True, exist_ok=True)

config_toml = textwrap.dedent('''\
    [models]
    default = "nrp:glm-5"
    recent = "nrp:glm-5"

    [models.providers.nrp]
    class_path = "langchain_openai:ChatOpenAI"
    models = [
        "glm-5",
    ]
    api_key_env = "ZAI_API_KEY"
    base_url = "https://api.z.ai/api/coding/paas/v4"

    [models.providers.nrp.params]
    temperature = 0
    stream_chunk_timeout = 1200.0
    max_retries = 6
''')
(deepagents_dir / 'config.toml').write_text(config_toml)

# Load the API key from Colab Secrets if we're in Colab, otherwise expect it preset.
try:
    from google.colab import userdata
    os.environ['ZAI_API_KEY'] = userdata.get('ZAI_API_KEY')
    print('Loaded ZAI_API_KEY from Colab Secrets')
except ImportError:
    if 'ZAI_API_KEY' not in os.environ:
        raise RuntimeError(
            'ZAI_API_KEY not set. In Colab, add it via the 🔑 sidebar; '
            'elsewhere, export it before launching the kernel.'
        )
    print('Using ZAI_API_KEY from existing environment')

print('config.toml written to', deepagents_dir / 'config.toml')

In [ ]:
# 4. Load ARGUS magics into the running IPython kernel
# This is the equivalent of putting sage_magic.py in ~/.ipython/profile_default/startup/
# on a normal install — Colab doesn't have that folder, so we load it manually.
#
# Use exec(..., globals()) instead of %run because Colab's %run runs in a
# temporary namespace that gets discarded — magic registration works (it's a
# side effect on the global IPython instance) but module-level names like
# SAGE_OUTPUT_DIR, SAGE_MESSAGES, _SAGE_MCP_TOOLS_BY_SERVER never reach the
# user namespace. exec(..., globals()) writes everything into the active
# notebook globals so all of ARGUS's internals are inspectable.
exec(open('/content/sage_magic.py').read(), globals())

## Smoke test

If everything above succeeded, the `%%ask` cell magic is now registered. The next cell asks ARGUS a trivial question — it should answer using the agent loop (an LLM call + maybe one tool call) and render the result inline.

In [ ]:
%%ask
What date is today?

## If the smoke test works

Next steps to validate:
- Try `%%mcp` against a public MCP server (e.g. `https://wenokn.fastmcp.app/mcp`) to confirm tool-calling works
- Try `%%skill github.com/klinucsd/sage_skills/tree/main/...` to install a skill at runtime
- Mount Google Drive (`from google.colab import drive; drive.mount('/content/drive')`) for persistent state across sessions

## If the smoke test fails

Common issues:
- **API key not set** — re-run cell 3 after adding the Colab Secret. The error message names the exact env var your config.toml expects (e.g. `ZAI_API_KEY`).
- **Magic command not found** — re-run cell 4; if `%%ask` still isn't recognized, the magic registration in `sage_magic.py` failed (check the cell-4 output for tracebacks)
- **Import errors during cell 4** — Python version mismatch or missing dependency; check `!python --version` (need 3.11+) and `!pip show deepagents-code`